# Media Framing Results Analysis

This notebook only analyzes already existing thesis result files.
It does not rebuild requests and it does not call the API.

Main outputs:
- run coverage summary
- overall label frequencies
- outlet-by-label counts
- outlet-by-label shares within each outlet


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd() / '2a_NER',
    Path.cwd(),
    Path.cwd().parent / '2a_NER',
    Path('/Users/MattisHaumann/Dev/Thesis/2a_NER'),
]
NOTEBOOK_DIR = next(
    (path for path in NOTEBOOK_DIR_CANDIDATES if (path / 'media_framing_batch_utils.py').exists()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate 2a_NER/media_framing_batch_utils.py from the current working directory.')

PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from media_framing_batch_utils import LEGACY_RESULT_COLUMNS, sort_by_source_order

FINAL_DIR = NOTEBOOK_DIR / 'outputs' / 'batch_media_framing' / 'thesis_final'
RUN_DIR = FINAL_DIR / 'normal_api_run'
ANALYSIS_DIR = FINAL_DIR / 'analysis'
RESULTS_PATH = RUN_DIR / 'media_framing_thesis_sync_results.csv'
ERRORS_PATH = RUN_DIR / 'media_framing_thesis_sync_errors.csv'
MANIFEST_PATH = RUN_DIR / 'media_framing_thesis_sync_manifest.csv'

ANALYSIS_SUMMARY_PATH = ANALYSIS_DIR / 'media_framing_thesis_analysis_summary.csv'
OVERALL_LABEL_SUMMARY_PATH = ANALYSIS_DIR / 'media_framing_thesis_overall_label_summary.csv'
OUTLET_LABEL_SUMMARY_PATH = ANALYSIS_DIR / 'media_framing_thesis_outlet_label_summary.csv'
OUTLET_LABEL_COUNTS_PIVOT_PATH = ANALYSIS_DIR / 'media_framing_thesis_outlet_label_counts_pivot.csv'
OUTLET_LABEL_SHARE_PIVOT_PATH = ANALYSIS_DIR / 'media_framing_thesis_outlet_label_share_pivot.csv'

CATEGORY_ORDER = [
    'POSITIONS-/PARTEILICHKEITS-BIAS',
    'VERZERRUNG/MANIPULATION',
    'DISINFORMATION/FALSCHDARSTELLUNG',
    'VERSAGEN/INKOMPETENZ',
    'NEUTRAL',
    'IRRELEVANT',
]

for required_path in [RESULTS_PATH, MANIFEST_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required analysis file not found: {required_path}')

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Results path: {RESULTS_PATH}')
print(f'Errors path: {ERRORS_PATH}')
print(f'Manifest path: {MANIFEST_PATH}')


## 1. Load and Validate the Existing Result Files


In [ ]:
results_df = pd.read_csv(RESULTS_PATH)
errors_df = pd.read_csv(ERRORS_PATH) if ERRORS_PATH.exists() and ERRORS_PATH.stat().st_size > 0 else pd.DataFrame()
manifest_df = pd.read_csv(MANIFEST_PATH)

if not results_df.empty:
    results_df = results_df[LEGACY_RESULT_COLUMNS].copy()
    results_df['source'] = results_df['source'].fillna('').astype(str)
    results_df['category'] = pd.Categorical(results_df['category'], categories=CATEGORY_ORDER, ordered=True)
    results_df = results_df.sort_values(
        ['row_id', 'context_idx', 'source', 'hit_text'],
        ascending=[True, True, True, True],
    ).drop_duplicates(subset=['hit_id'], keep='last').reset_index(drop=True)
    if results_df['hit_id'].duplicated().any():
        raise AssertionError('Duplicate hit_id values found in results_df.')

analysis_summary_df = pd.DataFrame(
    [
        {
            'manifest_rows': len(manifest_df),
            'result_rows': len(results_df),
            'error_rows': len(errors_df),
            'remaining_rows_without_success': max(len(manifest_df) - len(results_df), 0),
            'unique_articles_in_results': results_df['row_id'].nunique() if not results_df.empty else 0,
            'unique_outlets_in_results': results_df['source'].nunique() if not results_df.empty else 0,
        }
    ]
)
analysis_summary_df.to_csv(ANALYSIS_SUMMARY_PATH, index=False, encoding='utf-8')

print(f'Analysis summary written to: {ANALYSIS_SUMMARY_PATH}')
display(analysis_summary_df)
display(results_df.head(5))
display(errors_df.head(5))


## 2. Overall Label Frequencies


In [ ]:
if results_df.empty:
    print('No coded results available yet.')
else:
    overall_label_summary_df = (
        results_df.groupby('category', as_index=False, observed=False)
        .agg(
            coded_contexts=('hit_id', 'size'),
            unique_articles=('row_id', 'nunique'),
            unique_outlets=('source', 'nunique'),
            avg_hits_per_context=('count_hits', 'mean'),
            avg_unique_entities_per_context=('count_unique_entities', 'mean'),
        )
        .sort_values('category')
        .reset_index(drop=True)
    )
    overall_label_summary_df['share_pct'] = (
        overall_label_summary_df['coded_contexts'] / overall_label_summary_df['coded_contexts'].sum() * 100
    ).round(2)
    overall_label_summary_df['avg_hits_per_context'] = overall_label_summary_df['avg_hits_per_context'].round(2)
    overall_label_summary_df['avg_unique_entities_per_context'] = overall_label_summary_df['avg_unique_entities_per_context'].round(2)
    overall_label_summary_df.to_csv(OVERALL_LABEL_SUMMARY_PATH, index=False, encoding='utf-8')

    print(f'Overall label summary written to: {OVERALL_LABEL_SUMMARY_PATH}')
    display(overall_label_summary_df)


## 3. Compare Labels Across Outlets

The long table keeps both counts and within-outlet shares.
The pivot tables are easier to scan for thesis reporting.


In [ ]:
if results_df.empty:
    print('No coded results available yet.')
else:
    outlet_totals_df = (
        results_df.groupby('source', as_index=False)
        .agg(
            outlet_total_contexts=('hit_id', 'size'),
            outlet_total_articles=('row_id', 'nunique'),
            avg_hits_per_context=('count_hits', 'mean'),
            avg_unique_entities_per_context=('count_unique_entities', 'mean'),
        )
        .pipe(sort_by_source_order)
    )
    outlet_totals_df['avg_hits_per_context'] = outlet_totals_df['avg_hits_per_context'].round(2)
    outlet_totals_df['avg_unique_entities_per_context'] = outlet_totals_df['avg_unique_entities_per_context'].round(2)

    outlet_label_summary_df = (
        results_df.groupby(['source', 'category'], as_index=False, observed=False)
        .agg(
            coded_contexts=('hit_id', 'size'),
            unique_articles_with_label=('row_id', 'nunique'),
        )
        .merge(outlet_totals_df, on='source', how='left')
    )
    outlet_label_summary_df['share_within_outlet_pct'] = (
        outlet_label_summary_df.groupby('source')['coded_contexts']
        .transform(lambda values: (values / values.sum() * 100).round(2))
    )
    outlet_label_summary_df = (
        outlet_label_summary_df
        .pipe(sort_by_source_order)
        .sort_values(['source', 'category'])
        .reset_index(drop=True)
    )

    outlet_label_counts_pivot_df = (
        outlet_label_summary_df.pivot(index='source', columns='category', values='coded_contexts')
        .fillna(0)
        .astype(int)
        .reset_index()
        .pipe(sort_by_source_order)
    )
    outlet_label_share_pivot_df = (
        outlet_label_summary_df.pivot(index='source', columns='category', values='share_within_outlet_pct')
        .fillna(0)
        .reset_index()
        .pipe(sort_by_source_order)
    )

    outlet_label_summary_df.to_csv(OUTLET_LABEL_SUMMARY_PATH, index=False, encoding='utf-8')
    outlet_label_counts_pivot_df.to_csv(OUTLET_LABEL_COUNTS_PIVOT_PATH, index=False, encoding='utf-8')
    outlet_label_share_pivot_df.to_csv(OUTLET_LABEL_SHARE_PIVOT_PATH, index=False, encoding='utf-8')

    print(f'Outlet-label summary written to: {OUTLET_LABEL_SUMMARY_PATH}')
    print(f'Outlet-label count pivot written to: {OUTLET_LABEL_COUNTS_PIVOT_PATH}')
    print(f'Outlet-label share pivot written to: {OUTLET_LABEL_SHARE_PIVOT_PATH}')
    display(outlet_label_summary_df)
    display(outlet_label_counts_pivot_df)
    display(outlet_label_share_pivot_df)


## 4. Optional Inspection Table

Use this to inspect one outlet or one label without touching the run notebook.


In [ ]:
FILTER_SOURCE = None
FILTER_CATEGORY = None

if results_df.empty:
    print('No coded results available yet.')
else:
    inspection_df = results_df.copy()
    if FILTER_SOURCE:
        inspection_df = inspection_df[inspection_df['source'] == FILTER_SOURCE].copy()
    if FILTER_CATEGORY:
        inspection_df = inspection_df[inspection_df['category'] == FILTER_CATEGORY].copy()

    inspection_columns = [
        'row_id',
        'source',
        'Title',
        'hit_text',
        'category',
        'evidence',
        'count_hits',
        'count_unique_entities',
        'context_window',
    ]
    inspection_df = inspection_df[inspection_columns].sort_values(
        ['source', 'category', 'row_id', 'hit_text'],
        ascending=[True, True, True, True],
    ).reset_index(drop=True)

    print(f'Inspection rows: {len(inspection_df):,}')
    display(inspection_df.head(50))
